# Biomedical Classification with Machine Learning

**Author:** Mohamed Elsaid  
**Program:** M.S. Bioinformatics, Johns Hopkins University  
**Dataset:** Wisconsin Breast Cancer (Diagnostic) — `sklearn.datasets.load_breast_cancer`

---

## Project Overview

This notebook applies supervised machine learning to classify breast tumor biopsies as malignant or benign.
Three models are trained and compared:
- Logistic Regression
- Random Forest
- Support Vector Machine (SVM)

Evaluation metrics include accuracy, precision, recall, F1-score, ROC/AUC, and confusion matrices.
Scaling is applied inside sklearn Pipelines (fit on training data only) to prevent data leakage.

> **Disclaimer:** This is an academic portfolio project. Results are illustrative and not
> intended for clinical use.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
%matplotlib inline

print('Libraries loaded successfully.')

## 2. Load Data

In [ ]:
# Load built-in Breast Cancer Wisconsin dataset (no download required)
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = malignant, 1 = benign

print(f'Shape: {df.shape}')
print(f'Class distribution: {df["target"].value_counts().sort_index().to_dict()}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Basic info
print(df.info())
print('\nMissing values per column:')
print(df.isnull().sum())

In [ ]:
# Class distribution
TARGET = 'target'
labels = ['malignant (0)', 'benign (1)']
counts = df[TARGET].value_counts().sort_index()
counts.index = labels
counts.plot(kind='bar', color=['#A23B72', '#2E86AB'])
plt.title('Class Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Numeric summary statistics
df.describe().T

## 4. Data Cleaning

In [ ]:
# This dataset has no missing values — verify and proceed
missing = df.isnull().sum().sum()
print(f'Missing values: {missing}')
assert missing == 0, 'Unexpected missing values found'

## 5. Feature / Target Split

In [ ]:
# Target is already encoded: 0 = malignant, 1 = benign
y = df[TARGET]
X = df.drop(columns=[TARGET])
print(f'X shape: {X.shape}  |  y shape: {y.shape}')

## 6. Train / Test Split & Feature Scaling

In [ ]:
# Stratified split — scaling happens inside Pipelines during training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print('Note: StandardScaler is fit inside Pipelines (train data only) to prevent leakage.')

## 7. Model Training & Cross-Validation

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv, scoring=['accuracy', 'f1', 'roc_auc'], return_train_score=False
    )
    cv_results[name] = scores
    print(f"{name}: CV Acc = {scores['test_accuracy'].mean():.3f} "
          f"+/- {scores['test_accuracy'].std():.3f}, "
          f"AUC = {scores['test_roc_auc'].mean():.3f}")

In [ ]:
# Fit final models on full training split
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    print(f'{name} trained.')

## 8. Model Evaluation — Test Set

In [ ]:
for name, model in fitted.items():
    print(f'\n-- {name} --')
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=['malignant', 'benign']))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, model) in zip(axes, fitted.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=['malignant', 'benign']).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.savefig('../results/figures/confusion_matrices_all.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(6, 5))
colors = ['#2E86AB', '#A23B72', '#F18F01']
for (name, model), color in zip(fitted.items(), colors):
    y_score = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../results/figures/roc_curves_notebook.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Feature Importance (Random Forest)

In [ ]:
rf = fitted['Random Forest']
importances = rf.feature_importances_
feat_names = X.columns.tolist()
idx = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([feat_names[i] for i in idx][::-1], importances[idx][::-1], color='#2E86AB')
ax.set_xlabel('Gini Importance')
ax.set_title('Top 15 Features — Random Forest')
plt.tight_layout()
plt.savefig('../results/figures/feature_importance_rf_notebook.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Model Comparison Summary

In [ ]:
rows = []
for name, model in fitted.items():
    y_pred = model.predict(X_test)
    rows.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall':    recall_score(y_test, y_pred, average='weighted'),
        'F1':        f1_score(y_test, y_pred, average='weighted'),
    })

summary = pd.DataFrame(rows).set_index('Model').round(4)
summary

## 11. Biological / Scientific Interpretation

**Interpretation notes:**
- Top Random Forest features (`worst perimeter`, `worst concave points`, `worst radius`) align with known cytological markers of malignancy — larger, more irregular nuclei are associated with cancer.
- Logistic Regression and SVM achieved the highest test accuracy (~98%), likely because many features are near-linearly separable after scaling.
- Random Forest was slightly lower but still strong (>95%) and provides interpretable feature rankings.
- False negatives (missing a malignant case) are the clinically costlier error; all models showed high recall on the malignant class.

## 12. Limitations

- **No hyperparameter tuning:** Default or simple hyperparameters were used. A grid search or randomized search would likely improve performance.
- **Single dataset:** All results are specific to this dataset. Generalizability to other cohorts is unknown.
- **Feature importance ≠ causation:** Gini importance reflects predictive utility within this model, not biological causality.
- **Class imbalance:** If present, weighted metrics may mask poor performance on the minority class. SMOTE or class weighting would be appropriate next steps.
- **No external validation:** A truly independent test set from a different study would be needed to claim generalizability.

## 13. Conclusion

Logistic Regression and SVM tied for best test performance (98.3% accuracy, AUC ~0.995). Random Forest was close behind at 95.6% and identified `worst perimeter` and `worst concave points` as top predictors — consistent with cytology literature.

Next steps: hyperparameter tuning with `RandomizedSearchCV`, SHAP-based explainability, and validation on an independent cohort. This is a portfolio demonstration project, not a clinical tool.